# Pretrain Commutative CNN Encoder

Load the shared unlabeled pretraining dataset and save commutative CNN encoder weights for downstream classification notebooks.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from sklearn.model_selection import train_test_split

from src.ml import (
    CommutativeCNNClassifier,
    CommutativeCNNConfig,
    CommutativeCNNPretrainingConfig,
    LossWeightConfig,
    OptimizationConfig,
    augment_training_tensors_with_rotations,
    load_commutative_cnn_pretraining_config,
    write_commutative_cnn_pretraining_config,
)
from src.tensor_utils import load_unlabeled_tensor_dataset


In [2]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
pretrained_encoder_path = Path("artifacts/pretrained_commutative_cnn/encoder_state_v9.pt")
validation_fraction = 0.15
train_num_random_rotations = 0
rotation_range_degrees = 0.0

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(16, 32),
    spatial_kernel_size_z=(3, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(48, 64),
    temporal_st_kernel_sizes=(5, 3),
    temporal_ts_channels=(32, 48, 64),
    temporal_ts_kernel_sizes=(7, 5, 3),
    spatial_agg_channels=(32, 64),
    spatial_agg_kernel_size_z=(3, 1),
    spatial_agg_kernel_size_xy=(3, 3),
    spatial_agg_stride_z=(1, 1),
    spatial_agg_stride_xy=(1, 1),
    spatial_agg_pool_kernel_z=(1, 1),
    spatial_agg_pool_kernel_xy=(1, 2),
    spatial_agg_pool_stride_z=(1, 1),
    spatial_agg_pool_stride_xy=(1, 2),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=64,
    num_prototypes=64,
    probe_region_grid=(1, 2, 2),
    probe_time_bins=8,
    probe_frequency_bins=4,
    dropout=0.25,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=70,
    learning_rate=7.5e-5,
    weight_decay=7.5e-4,
    early_stopping_patience=8,
    early_stopping_min_delta=5e-5,
    early_stopping_start_epoch=18,
    early_stopping_monitor="self_probe_loss",
    early_stopping_smoothing="median",
    early_stopping_smoothing_window=3,
    training_plot_dir="artifacts/pretrained_commutative_cnn/loss_plots",
    training_plot_every_n_epochs=2,
    training_plot_smoothing_window=5,
    scheduler_patience=3,
    scheduler_factor=0.5,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    lambda_cross=0.02,
    cross_warmup_epochs=20,
    cross_ramp_epochs=50,
    prototype_temperature=0.25,
    prototype_alignment_weight=0.0,
    prototype_warmup_epochs=16,
    prototype_ramp_epochs=36,
    latent_alignment_weight=0.0,
    lambda_align=0.0,
    probe_mask_probability=1.0,
    probe_alpha_local=1.0,
    probe_alpha_region_time=1.0,
    probe_alpha_derivative=0.25,
    probe_alpha_frequency=0.10,
    probe_alpha_correlation=0.05,
)

pretraining_config = CommutativeCNNPretrainingConfig(
    unlabeled_dataset_path=unlabeled_dataset_path,
    pretrained_encoder_path=pretrained_encoder_path,
    validation_fraction=validation_fraction,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
pretraining_config_path = write_commutative_cnn_pretraining_config(pretraining_config)
pretraining_config = load_commutative_cnn_pretraining_config(pretraining_config_path)
print(f"Commutative CNN pretraining config in {pretraining_config_path}")
print(f"Pretraining loss PDFs: {Path(optimization_config.training_plot_dir).resolve()}")
print(f"Pretrained encoder checkpoint target: {pretrained_encoder_path.resolve()}")
pretraining_config


Commutative CNN pretraining config in /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/config.yaml
Pretraining loss PDFs: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/loss_plots
Pretrained encoder checkpoint target: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/encoder_state_v9.pt


CommutativeCNNPretrainingConfig(unlabeled_dataset_path=PosixPath('.dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks'), pretrained_encoder_path=PosixPath('artifacts/pretrained_commutative_cnn/encoder_state_v9.pt'), validation_fraction=0.15, train_num_random_rotations=0, rotation_range_degrees=0.0, model_config=CommutativeCNNConfig(spatial_conv_channels=(16, 32), spatial_kernel_size_z=(3, 1), spatial_kernel_size_xy=(5, 3), spatial_stride_z=(1, 1), spatial_stride_xy=(1, 1), spatial_pool_kernel_z=(1, 1), spatial_pool_kernel_xy=(2, 2), spatial_pool_stride_z=(1, 1), spatial_pool_stride_xy=(2, 2), temporal_st_channels=(48, 64), temporal_st_kernel_sizes=(5, 3), temporal_ts_channels=(32, 48, 64), temporal_ts_kernel_sizes=(7, 5, 3), spatial_agg_channels=(32, 64), spatial_agg_kernel_size_z=(3, 1), spatial_agg_kernel_size_xy=(3, 3), spatial_agg_stride_z=(1, 1), spatial_agg_stride_xy=(1, 1), spatial_agg_pool_kernel_z=(1, 1), spatial_agg_pool_kernel_xy=(1, 2), spatial_agg_pool_strid

In [3]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
train_indices, val_indices = train_test_split(
    range(len(unlabeled_dataset["tensors"])),
    test_size=validation_fraction,
    random_state=optimization_config.random_state,
    shuffle=True,
)
X_train_base = unlabeled_dataset["tensors"][train_indices]
X_val = unlabeled_dataset["tensors"][val_indices]
metadata_train_base = unlabeled_dataset["metadata"].iloc[train_indices].reset_index(drop=True)
metadata_val = unlabeled_dataset["metadata"].iloc[val_indices].reset_index(drop=True)
X_train, _, metadata_train = augment_training_tensors_with_rotations(
    X_train_base,
    [0] * len(X_train_base),
    metadata=metadata_train_base,
    num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)
{
    "all_tensors": unlabeled_dataset["tensors"].shape,
    "all_metadata": unlabeled_dataset["metadata"].shape,
    "train_base_tensors": X_train_base.shape,
    "train_tensors": X_train.shape,
    "val_tensors": X_val.shape,
    "train_base_metadata": metadata_train_base.shape,
    "train_metadata": metadata_train.shape,
    "val_metadata": metadata_val.shape,
}


{'all_tensors': torch.Size([2144, 20, 5, 96, 96]),
 'all_metadata': (2144, 7),
 'train_base_tensors': torch.Size([1822, 20, 5, 96, 96]),
 'train_tensors': torch.Size([1822, 20, 5, 96, 96]),
 'val_tensors': torch.Size([322, 20, 5, 96, 96]),
 'train_base_metadata': (1822, 7),
 'train_metadata': (1822, 7),
 'val_metadata': (322, 7)}

## Output Review

The completed `v8` run selected `best_epoch=036` because early stopping only began at epoch `036`, but raw validation was already unstable there (`val_loss=90.4292`). The best raw validation was much earlier at epoch `025` (`val_loss=1.9319`), and validation self-probe loss then diverged while training self-probe loss stayed low. The next run writes `encoder_state_v9.pt`, lowers the optimizer to `learning_rate=7.5e-5`, raises dropout and weight decay, monitors `val_self_probe_loss` from epoch `018`, and uses full probe masks during training to match validation. Teacher-student cross pressure is reduced to `lambda_cross=0.02` with a slower warmup/ramp, and prototype alignment is disabled for this pass so the checkpoint is selected on stable coarse probe reconstruction rather than noisy auxiliary alignment.

In [ ]:
%%time
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(X_train, validation_data=X_val)
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretrained_encoder_path


Commutative CNN model: parameters=182,603 trainable=182,603 size=0.70 MB
cols:
    ep=epoch
    lr=learning_rate
    eta=estimated_time_remaining
    trL=train_loss
    trS=train_self_probe_loss
    trX=train_cross_probe_loss
    trP=train_prototype_alignment_loss
    trLA=train_latent_alignment_loss
     ep       lr       eta |      trL      trS      trX      trP     trLA |      vaL      vaS      vaX      vaP     vaLA
001/070 7.50e-05  14:27:15 |   2.5006   2.5006   0.0920   4.4815   0.3359 |   5.7398   5.7398   2.7197   4.9018   8.2053
002/070 7.50e-05  13:07:10 |   2.4149   2.4149   0.1192   4.4745   0.4590 |  10.2667  10.2667  13.1677   4.9801  29.6085
003/070 7.50e-05  12:18:28 |   2.3575   2.3575   0.1239   4.3586   0.4443 |   5.5864   5.5864   0.4199   4.3337   0.8970
004/070 7.50e-05  11:46:41 |   2.3356   2.3356   0.1360   4.2875   0.4310 |   4.7356   4.7356   1.3958   4.2812   1.9448
005/070 7.50e-05  12:08:51 |   2.3069   2.3069   0.1498   4.2402   0.4345 |  19.5500  19.5500

In [ ]:
model.pretrain_history_.tail()